# GFN National Footprint and Biocapacity Accounts via the API (v3)

Changes from `GFN_amended.ipynb`:
1. **Credentials fixed.** `userdata.get()` takes the *name* of a Colab secret, not its value. Add two
   secrets in the key-icon panel: `GFN_API_KEY` (your key) and `GFN_API_USER` (your e-mail). Never
   paste the key into a cell. **Regenerate the key that was pasted into the previous notebook.**
2. **Per-country fallback.** The bulk call `data/all/all/<type>` returned EF for only 128 countries
   (Algeria, Morocco, Iran, Kazakhstan, Saudi Arabia, Namibia, Armenia, Georgia, Hong Kong ... are
   absent). For every country listed by `/countries` but missing from the bulk result, the notebook
   asks `data/<code>/all/<type>` and logs the exact HTTP status and message, so we learn whether the
   data exist but are withheld (licence or quality threshold) or do not exist.
3. **Per-year check.** One bulk call per year for a test year shows whether `all/all` truncates.
4. Everything else as before: raw download saved unchanged, panel keyed on the GFN country code,
   Namibia's `"NA"` kept, quality scores kept, `accounts` flag for nowcast years.

In [ ]:
import time, json, datetime as dt
import requests
import pandas as pd

try:                                   # Google Colab: Secrets panel (key icon)
    from google.colab import userdata
    KEY, USER = userdata.get("GFN_API_KEY"), userdata.get("GFN_API_USER")
except Exception:                      # elsewhere: environment variables, or a hidden prompt
    import os, getpass
    KEY = os.environ.get("GFN_API_KEY") or getpass.getpass("GFN API key: ")
    USER = os.environ.get("GFN_API_USER") or input("GFN API user (e-mail): ")

BASE = "https://api.footprintnetwork.org/v1"
AUTH = (USER, KEY)
DOWNLOAD_DATE = dt.date.today().isoformat()
RECORDS = ["EFCpc", "BCpc", "EFCtot", "BCtot", "pop"]
LOG = []                               # every non-200 answer, for the diagnostics below

In [ ]:
def get(path, tries=4):
    """GET an API path; return (DataFrame or None, status, message). Retries transient errors."""
    last = (None, None, "")
    for k in range(tries):
        try:
            r = requests.get(f"{BASE}/{path}", auth=AUTH, timeout=120)
            if r.status_code == 200:
                js = r.json()
                return (pd.DataFrame(js) if js else pd.DataFrame()), 200, ""
            last = (None, r.status_code, r.text[:300])
            if r.status_code in (400, 401, 403, 404):
                break                  # not transient: do not retry
        except requests.RequestException as e:
            last = (None, None, repr(e))
        time.sleep(2 ** k)
    LOG.append({"path": path, "status": last[1], "message": last[2]})
    return last

countries, st, msg = get("countries")
assert st == 200, (st, msg)
types, _, _ = get("types")
years, _, _ = get("years")
print(countries.shape, "countries listed by the API")
print(types[["code", "name"]].to_string(index=False) if types is not None else "no types")

In [ ]:
# 1) Bulk download, as before
frames = []
for rec in RECORDS:
    df, st, msg = get(f"data/all/all/{rec}")
    print(rec, st, None if df is None else df.shape, msg[:120])
    if df is not None and not df.empty:
        frames.append(df.assign(type_code=rec, source="bulk"))
    time.sleep(1)
bulk = pd.concat(frames, ignore_index=True)
have = set(bulk.loc[(bulk["record"] == "EFConsPerCap") & bulk["value"].notna(), "countryCode"])
code_col = "countryCode" if "countryCode" in countries.columns else countries.columns[0]
gap = countries[~countries[code_col].isin(have)]
print(len(have), "countries with EF in bulk;", len(gap), "listed but missing")

In [ ]:
# 2) Per-country fallback for the missing ones (about 5 calls per country; pauses to be polite)
extra = []
for _, c in gap.iterrows():
    code_ = c[code_col]
    for rec in RECORDS:
        df, st, msg = get(f"data/{code_}/all/{rec}")
        if df is not None and not df.empty:
            extra.append(df.assign(type_code=rec, source="per-country"))
        time.sleep(0.5)
extra = pd.concat(extra, ignore_index=True) if extra else pd.DataFrame()
got = set(extra.loc[(extra.get("record") == "EFConsPerCap") & extra["value"].notna(), "countryCode"]) if len(extra) else set()
print(len(got), "additional countries recovered with EF values")

In [ ]:
# 3) Diagnostics: what the API says about the countries still missing
log = pd.DataFrame(LOG)
print("non-200 answers by status:")
print(log["status"].value_counts(dropna=False).to_string() if len(log) else "none")
print(log.head(15).to_string() if len(log) else "")
still = gap[~gap[code_col].isin(got)]
print(len(still), "countries still without EF:")
print(still.to_string())
# per-year bulk check for one recent accounts year (does all/all truncate?)
df, st, msg = get("data/all/2020/EFCpc")
n2020 = df.loc[df["value"].notna(), "countryCode"].nunique() if df is not None else None
print("countries with EF in data/all/2020/EFCpc:", n2020, st, msg[:120])

In [ ]:
# 4) Save the raw long file unchanged, then the tidy panel (keyed on GFN country code)
LAST_ACCOUNTS_YEAR = 2022   # <-- check the edition's documentation
long = pd.concat([bulk, extra], ignore_index=True)
long.to_csv(f"gfn_long_raw_{DOWNLOAD_DATE}.csv", index=False)
log.to_csv(f"gfn_api_log_{DOWNLOAD_DATE}.csv", index=False)

key = ["countryCode", "year"]
vals = (long.pivot_table(index=key, columns="record", values="value", aggfunc="first")
            .rename(columns={"EFConsPerCap": "ef_pc", "BiocapPerCap": "bc_pc",
                             "EFConsTotGHA": "ef_tot", "BiocapTotGHA": "bc_tot", "Population": "pop"}))
scores = (long[long["record"].isin(["EFConsPerCap", "BiocapPerCap"])]
            .pivot_table(index=key, columns="record", values="score", aggfunc="first")
            .rename(columns={"EFConsPerCap": "score_ef", "BiocapPerCap": "score_bc"}))
src = long.groupby(key)["source"].first().rename("source")
names = (long[["countryCode", "countryName", "isoa2"]].drop_duplicates("countryCode")
            .rename(columns={"countryCode": "gfn_code", "countryName": "country", "isoa2": "iso2"}))
panel = (vals.join(scores, how="left").join(src).reset_index()
             .rename(columns={"countryCode": "gfn_code"}).merge(names, on="gfn_code", how="left"))
panel["accounts"] = (panel["year"] <= LAST_ACCOUNTS_YEAR).astype(int)
cols = ["iso2", "gfn_code", "country", "year", "ef_pc", "bc_pc", "ef_tot", "bc_tot", "pop",
        "score_ef", "score_bc", "accounts", "source"]
panel = panel.reindex(columns=cols).sort_values(["gfn_code", "year"])
panel.to_csv("gfn_api.csv", index=False, na_rep="")      # "NA" (Namibia) stays text
json.dump({"download_date": DOWNLOAD_DATE, "api": BASE, "records": RECORDS,
           "last_accounts_year": LAST_ACCOUNTS_YEAR, "countries": int(panel["gfn_code"].nunique()),
           "countries_with_ef": int(panel.loc[panel["ef_pc"].notna(), "gfn_code"].nunique())},
          open("gfn_api_metadata.json", "w"), indent=2)
print(panel.shape, panel.loc[panel["ef_pc"].notna(), "gfn_code"].nunique(), "countries with EF")
# Upload gfn_api.csv AND gfn_api_log_<date>.csv: the log tells us why countries are missing.